In [1]:
# ============================================================
# SECTION 0 — GLOBAL CONFIGURATION
# Run this cell first in every session
# ============================================================

# ── Core imports ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import random
import json
import os
import warnings
warnings.filterwarnings('ignore')

# ── Deep learning ────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Input, Dense, Dropout, LSTM,
                                     Bidirectional, Conv1D, MaxPooling1D,
                                     Flatten, Concatenate)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

# ── ML models ────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve, confusion_matrix)

# ── Visualization ────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from math import radians, sin, cos, sqrt, atan2

# ── Reproducibility ──────────────────────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

# ── Drive paths ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"Drive mounted. Project folder: {DRIVE_PATH}")

# ── Haversine distance function ───────────────────────────────
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# ── Evaluation helpers ────────────────────────────────────────
def fpr_constrained_recall(y_true, y_prob, fpr_threshold):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    valid_idx = np.where(fpr <= fpr_threshold)[0]
    if len(valid_idx) == 0:
        return 0.0
    return tpr[valid_idx[np.argmax(tpr[valid_idx])]]

def evaluate_model(name, y_true, y_pred, y_prob):
    auc         = roc_auc_score(y_true, y_prob)
    f1          = f1_score(y_true, y_pred)
    precision   = precision_score(y_true, y_pred)
    recall      = recall_score(y_true, y_pred)
    r1fpr       = fpr_constrained_recall(y_true, y_prob, 0.01)
    r3fpr       = fpr_constrained_recall(y_true, y_prob, 0.03)
    print(f"\n=== {name} ===")
    print(f"AUC:              {auc:.4f}")
    print(f"F1:               {f1:.4f}")
    print(f"Precision:        {precision:.4f}")
    print(f"Recall:           {recall:.4f}")
    print(f"Recall @ 1% FPR:  {r1fpr:.4f}")
    print(f"Recall @ 3% FPR:  {r3fpr:.4f}")
    result = {'name': name, 'auc': auc, 'f1': f1, 'precision': precision,
              'recall': recall, 'recall_1fpr': r1fpr, 'recall_3fpr': r3fpr}
    # Auto-save to Drive
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    all_results = [r for r in all_results if r['name'] != name]
    all_results.append(result)
    json.dump(all_results, open(results_path, 'w'), indent=2)
    print("Result saved to Drive.")
    return result

def print_results_table():
    results_path = f'{DRIVE_PATH}/results.json'
    if not os.path.exists(results_path):
        print("No results saved yet.")
        return
    all_results = json.load(open(results_path))
    print(f"\n{'Model':<35} {'AUC':>6} {'F1':>6} {'Recall':>8} {'R@1%FPR':>9} {'R@3%FPR':>9}")
    print("-" * 75)
    for r in all_results:
        print(f"{r['name']:<35} {r['auc']:>6.4f} {r['f1']:>6.4f} "
              f"{r['recall']:>8.4f} {r['recall_1fpr']:>9.4f} {r['recall_3fpr']:>9.4f}")

# ── Class weight helper ───────────────────────────────────────
def get_class_weights(y):
    weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y)
    return {0: weights[0], 1: weights[1]}

print("\n✅ Section 0 complete — all libraries loaded, seeds set.")
print(f"TensorFlow version: {tf.__version__}")
print(f"Random seed: {RANDOM_SEED}")

Mounted at /content/drive
Drive mounted. Project folder: /content/drive/MyDrive/HOBA_Fraud_Detection_V2

✅ Section 0 complete — all libraries loaded, seeds set.
TensorFlow version: 2.20.0
Random seed: 42


In [ ]:
# ============================================================
# SECTION 13 — CNN — FULLY SELF-CONTAINED
# ============================================================
import gc, os, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve)
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Dense, Dropout, Input, Conv1D,
                                     MaxPooling1D, Flatten, Reshape)
from tensorflow.keras.callbacks import EarlyStopping

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

# ── Evaluation helpers ────────────────────────────────────────
def get_metrics_at_fpr(y_true, y_prob, fpr_threshold):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    valid_idx = np.where(fpr <= fpr_threshold)[0]
    if len(valid_idx) == 0:
        return {'f1': 0, 'precision': 0, 'recall': 0, 'accuracy': 0}
    best_idx = valid_idx[np.argmax(tpr[valid_idx])]
    thresh = thresholds[best_idx]
    y_pred_thresh = (y_prob >= thresh).astype(int)
    TP = ((y_pred_thresh==1)&(y_true==1)).sum()
    TN = ((y_pred_thresh==0)&(y_true==0)).sum()
    FP = ((y_pred_thresh==1)&(y_true==0)).sum()
    FN = ((y_pred_thresh==0)&(y_true==1)).sum()
    precision = TP/(TP+FP) if (TP+FP)>0 else 0
    recall    = TP/(TP+FN) if (TP+FN)>0 else 0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
    accuracy  = (TP+TN)/len(y_true)
    return {'f1': f1, 'precision': precision, 'recall': recall, 'accuracy': accuracy}

def evaluate_model(name, y_true, y_pred, y_prob):
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    acc  = (y_pred==y_true).sum()/len(y_true)
    m1   = get_metrics_at_fpr(y_true, y_prob, 0.01)
    m3   = get_metrics_at_fpr(y_true, y_prob, 0.03)
    print(f"\n=== {name} ===")
    print(f"Overall  — AUC:{auc:.4f} F1:{f1:.4f} Pre:{prec:.4f} Rec:{rec:.4f} Acc:{acc:.4f}")
    print(f"@ 1% FPR — F1:{m1['f1']:.4f} Pre:{m1['precision']:.4f} Rec:{m1['recall']:.4f} Acc:{m1['accuracy']:.4f}")
    print(f"@ 3% FPR — F1:{m3['f1']:.4f} Pre:{m3['precision']:.4f} Rec:{m3['recall']:.4f} Acc:{m3['accuracy']:.4f}")
    result = {
        'name': name, 'auc': auc, 'f1': f1, 'precision': prec,
        'recall': rec, 'accuracy': acc,
        'f1_1fpr': m1['f1'], 'precision_1fpr': m1['precision'],
        'recall_1fpr': m1['recall'], 'accuracy_1fpr': m1['accuracy'],
        'f1_3fpr': m3['f1'], 'precision_3fpr': m3['precision'],
        'recall_3fpr': m3['recall'], 'accuracy_3fpr': m3['accuracy']
    }
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    all_results = [r for r in all_results if r['name'] != name]
    all_results.append(result)
    json.dump(all_results, open(results_path, 'w'), indent=2)
    print("Result saved to Drive.")
    return result

def print_results_table():
    results_path = f'{DRIVE_PATH}/results.json'
    if not os.path.exists(results_path):
        print("No results saved yet.")
        return
    all_results = json.load(open(results_path))
    print(f"\n{'Model':<25} {'AUC':>6} {'F1':>6} {'Rec':>6} {'Acc':>6} | {'F1@1%':>6} {'Pre@1%':>7} {'Rec@1%':>7} | {'F1@3%':>6} {'Rec@3%':>7}")
    print("-"*105)
    for r in all_results:
        print(f"{r['name']:<25} {r['auc']:>6.4f} {r['f1']:>6.4f} {r['recall']:>6.4f} {r.get('accuracy',0):>6.4f} | "
              f"{r.get('f1_1fpr',0):>6.4f} {r.get('precision_1fpr',0):>7.4f} {r.get('recall_1fpr',0):>7.4f} | "
              f"{r.get('f1_3fpr',0):>6.4f} {r.get('recall_3fpr',0):>7.4f}")

# ── Load RFM ─────────────────────────────────────────────────
print("\nLoading RFM...")
train_rfm_df = pd.read_parquet(f'{DRIVE_PATH}/train_rfm.parquet')
test_rfm_df  = pd.read_parquet(f'{DRIVE_PATH}/test_rfm.parquet')
rfm_cols = [c for c in train_rfm_df.columns if c.startswith('rfm_')]
y_train_rfm = train_rfm_df['is_fraud'].astype(int).to_numpy()
y_test_rfm  = test_rfm_df['is_fraud'].astype(int).to_numpy()
X_train_rfm_raw = train_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
X_test_rfm_raw  = test_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
del train_rfm_df, test_rfm_df; gc.collect()

scaler_rfm = StandardScaler()
X_train_rfm = scaler_rfm.fit_transform(X_train_rfm_raw)
X_test_rfm  = scaler_rfm.transform(X_test_rfm_raw)
del X_train_rfm_raw, X_test_rfm_raw; gc.collect()
print(f"RFM ready: {X_train_rfm.shape} | RAM: {psutil.virtual_memory().percent}%")

weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train_rfm)
class_weight_dict = {0: float(weights[0]), 1: float(weights[1])}
early_stop = EarlyStopping(monitor='val_loss', patience=3,
                            restore_best_weights=True, verbose=1)

# ── Build CNN ─────────────────────────────────────────────────
def build_cnn(input_dim):
    model = Sequential([
        Reshape((input_dim, 1), input_shape=(input_dim,)),
        Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Conv1D(32, kernel_size=3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

# ── CNN on RFM ────────────────────────────────────────────────
print("\nTraining CNN on RFM features...")
cnn_rfm = build_cnn(input_dim=X_train_rfm.shape[1])
cnn_rfm.fit(X_train_rfm, y_train_rfm, epochs=20, batch_size=512,
            validation_split=0.1, class_weight=class_weight_dict,
            callbacks=[early_stop], verbose=1)
y_prob = cnn_rfm.predict(X_test_rfm, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)
evaluate_model("CNN — RFM", y_test_rfm, y_pred, y_prob)
print(f"RAM after CNN-RFM: {psutil.virtual_memory().percent}%")
del cnn_rfm, y_pred, y_prob, X_train_rfm, X_test_rfm; gc.collect()

# ── HOBA streaming setup ──────────────────────────────────────
print("\nSetting up HOBA streaming...")
pf_train = pq.ParquetFile(f'{DRIVE_PATH}/train_hoba_scaled.parquet')
hoba_cols = [c for c in pf_train.schema_arrow.names if c.startswith('hoba_')]
n_cols = len(hoba_cols)
n_rows_train = pf_train.metadata.num_rows
pf_test_f = pq.ParquetFile(f'{DRIVE_PATH}/test_hoba_scaled.parquet')
n_rows_test = pf_test_f.metadata.num_rows

VAL_FRACTION = 0.1
n_val   = int(n_rows_train * VAL_FRACTION)
n_train = n_rows_train - n_val

labels = np.concatenate([
    batch.column('is_fraud').to_numpy(zero_copy_only=False)
    for batch in pf_train.iter_batches(batch_size=200_000, columns=['is_fraud'])
])
weights_h = compute_class_weight('balanced', classes=np.array([0,1]), y=labels)
cw_hoba = {0: float(weights_h[0]), 1: float(weights_h[1])}
del labels; gc.collect()
print(f"RAM: {psutil.virtual_memory().percent}%")

BATCH_SIZE = 512
train_path = f'{DRIVE_PATH}/train_hoba_scaled.parquet'
test_path  = f'{DRIVE_PATH}/test_hoba_scaled.parquet'

def make_generator(parquet_path, row_start, row_end, batch_size, cols):
    def gen():
        pf = pq.ParquetFile(parquet_path)
        row_cursor = 0
        for batch in pf.iter_batches(batch_size=batch_size, columns=cols+['is_fraud']):
            n = batch.num_rows
            lo = max(row_start, row_cursor)
            hi = min(row_end, row_cursor+n)
            row_cursor += n
            if lo >= hi:
                continue
            olo, ohi = lo-(row_cursor-n), hi-(row_cursor-n)
            X = np.column_stack([batch.column(c).to_numpy(zero_copy_only=False)[olo:ohi]
                                  for c in cols]).astype(np.float32)
            y = batch.column('is_fraud').to_numpy(zero_copy_only=False)[olo:ohi].astype(np.int8)
            yield X, y
    return gen

sig = (tf.TensorSpec(shape=(None, n_cols), dtype=tf.float32),
       tf.TensorSpec(shape=(None,),        dtype=tf.int8))

train_ds = tf.data.Dataset.from_generator(
    make_generator(train_path, 0, n_train, BATCH_SIZE, hoba_cols),
    output_signature=sig).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_generator(
    make_generator(train_path, n_train, n_rows_train, BATCH_SIZE, hoba_cols),
    output_signature=sig).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_generator(
    make_generator(test_path, 0, n_rows_test, BATCH_SIZE, hoba_cols),
    output_signature=sig).prefetch(tf.data.AUTOTUNE)

# ── CNN on HOBA ───────────────────────────────────────────────
print("\nTraining CNN on HOBA features...")
cnn_hoba = build_cnn(input_dim=n_cols)
cnn_hoba.fit(train_ds, validation_data=val_ds, epochs=20,
             class_weight=cw_hoba, callbacks=[early_stop], verbose=1)
print(f"RAM after CNN-HOBA fit: {psutil.virtual_memory().percent}%")

print("\nPredicting on test set...")
y_prob_list, y_test_list = [], []
for X_batch, y_batch in test_ds:
    y_prob_list.append(cnn_hoba.predict(X_batch, verbose=0).flatten())
    y_test_list.append(y_batch.numpy())
y_prob = np.concatenate(y_prob_list)
y_test = np.concatenate(y_test_list)
y_pred = (y_prob >= 0.5).astype(int)
evaluate_model("CNN — HOBA", y_test, y_pred, y_prob)
print(f"RAM after eval: {psutil.virtual_memory().percent}%")

print("\n")
print_results_table()

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Loading RFM...
RFM ready: (1035957, 65) | RAM: 23.8%

Training CNN on RFM features...
Epoch 1/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - AUC: 0.9159 - loss: 0.3712 - val_AUC: 0.9621 - val_loss: 0.2692
Epoch 2/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - AUC: 0.9470 - loss: 0.2921 - val_AUC: 0.9699 - val_loss: 0.2473
Epoch 3/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - AUC: 0.9549 - loss: 0.2677 - val_AUC: 0.9738 - val_loss: 0.2280
Epoch 4/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - AUC: 0.9595 - loss: 0.2546 - val_AUC: 0.9745 - val_loss: 0.1898
Epoch 5/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - AUC: 0.9613 - loss: 0.2479 - val_AUC: 0.9764 - val_loss: 0.2070
Epoch 6/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - AUC: 0.9645 - loss: 0.2377 - val_AUC: 0.9765 - val_loss: 0.2065
Epoch 7/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - AUC: 0.9645 - loss: 0.2352 - val_AUC: 0.9783 - val_loss: 0

# Train RNN on RFM Features

This cell trains an LSTM on the RFM feature vector.

The 65 engineered RFM features are treated as a one-dimensional sequence,
allowing the recurrent network to learn dependencies among behavioral features.

In [ ]:
# ============================================================
# CELL 1 — Imports, Configuration & Helper Functions
# ============================================================

import gc
import os
import json

import gc
import json
import os

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil

import tensorflow as tf

import numpy as np
import pandas as pd
import psutil

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score
)

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Reshape,
    LSTM,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# Configuration
# ============================================================

DRIVE_PATH = "/content/drive/MyDrive/HOBA_Fraud_Detection_V2"

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("GPU:", tf.config.list_physical_devices("GPU"))
print(f"Current RAM: {psutil.virtual_memory().percent}%")


# ============================================================
# Evaluation Functions
# ============================================================

def get_metrics_at_fpr(y_true, y_prob, fpr_threshold):

    fpr, tpr, thresholds = roc_curve(y_true, y_prob)

    valid_idx = np.where(fpr <= fpr_threshold)[0]

    if len(valid_idx) == 0:
        return {
            'f1':0,
            'precision':0,
            'recall':0,
            'accuracy':0
        }

    best_idx = valid_idx[np.argmax(tpr[valid_idx])]
    thresh = thresholds[best_idx]

    y_pred = (y_prob >= thresh).astype(int)

    TP=((y_pred==1)&(y_true==1)).sum()
    TN=((y_pred==0)&(y_true==0)).sum()
    FP=((y_pred==1)&(y_true==0)).sum()
    FN=((y_pred==0)&(y_true==1)).sum()

    precision = TP/(TP+FP) if TP+FP else 0
    recall    = TP/(TP+FN) if TP+FN else 0
    f1        = 2*precision*recall/(precision+recall) if precision+recall else 0

    return {
        'f1':f1,
        'precision':precision,
        'recall':recall,
        'accuracy':(TP+TN)/len(y_true)
    }


def evaluate_model(name, y_true, y_pred, y_prob):

    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    acc  = (y_true==y_pred).mean()

    m1 = get_metrics_at_fpr(y_true, y_prob, 0.01)
    m3 = get_metrics_at_fpr(y_true, y_prob, 0.03)

    print(f"\n=== {name} ===")
    print(f"Overall  — AUC:{auc:.4f} F1:{f1:.4f} Pre:{prec:.4f} Rec:{rec:.4f} Acc:{acc:.4f}")
    print(f"@ 1% FPR — F1:{m1['f1']:.4f} Pre:{m1['precision']:.4f} Rec:{m1['recall']:.4f} Acc:{m1['accuracy']:.4f}")
    print(f"@ 3% FPR — F1:{m3['f1']:.4f} Pre:{m3['precision']:.4f} Rec:{m3['recall']:.4f} Acc:{m3['accuracy']:.4f}")

    result = {
        'name':name,
        'auc':auc,
        'f1':f1,
        'precision':prec,
        'recall':rec,
        'accuracy':acc,
        'f1_1fpr':m1['f1'],
        'precision_1fpr':m1['precision'],
        'recall_1fpr':m1['recall'],
        'accuracy_1fpr':m1['accuracy'],
        'f1_3fpr':m3['f1'],
        'precision_3fpr':m3['precision'],
        'recall_3fpr':m3['recall'],
        'accuracy_3fpr':m3['accuracy']
    }

    results_path = f"{DRIVE_PATH}/results.json"

    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []

    all_results = [r for r in all_results if r["name"] != name]

    all_results.append(result)

    json.dump(all_results, open(results_path, "w"), indent=2)

    print("Result saved to Drive.")

    return result


print("\n✓ Cell 1 Loaded Successfully")

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Current RAM: 15.4%

✓ Cell 1 Loaded Successfully


In [ ]:
# ============================================================
# Load RFM
# ============================================================

print("\nLoading RFM...")

train_rfm = pd.read_parquet(f"{DRIVE_PATH}/train_rfm.parquet")
test_rfm  = pd.read_parquet(f"{DRIVE_PATH}/test_rfm.parquet")

rfm_cols = [c for c in train_rfm.columns if c.startswith("rfm_")]

X_train = train_rfm[rfm_cols].values.astype(np.float32)
X_test  = test_rfm[rfm_cols].values.astype(np.float32)

y_train = train_rfm["is_fraud"].astype(int).values
y_test  = test_rfm["is_fraud"].astype(int).values

del train_rfm, test_rfm
gc.collect()

print(f"RFM ready: {X_train.shape}")
print(f"RAM: {psutil.virtual_memory().percent}%")

# ============================================================
# Class Weights
# ============================================================

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1]),
    y=y_train
)

class_weight_dict = {
    0: float(weights[0]),
    1: float(weights[1])
}

# ============================================================
# Build RNN
# ============================================================

model = Sequential([

    Reshape((len(rfm_cols),1), input_shape=(len(rfm_cols),)),

    LSTM(
        64,
        return_sequences=False
    ),

    Dropout(0.30),

    Dense(
        32,
        activation="relu"
    ),

    Dropout(0.20),

    Dense(
        1,
        activation="sigmoid"
    )

])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["AUC"]
)

early = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("\nTraining RNN on RFM features...")

history = model.fit(

    X_train,
    y_train,

    epochs=20,

    batch_size=512,

    validation_split=0.20,

    class_weight=class_weight_dict,

    callbacks=[early],

    verbose=1
)

# ============================================================
# Evaluate
# ============================================================

y_prob = model.predict(
    X_test,
    batch_size=4096,
    verbose=1
).ravel()

y_pred = (y_prob>=0.5).astype(int)

evaluate_model(
    "RNN — RFM",
    y_test,
    y_pred,
    y_prob
)

del X_train
del X_test
del y_train
del y_test
gc.collect()

print(f"RAM after RNN-RFM: {psutil.virtual_memory().percent}%")


Loading RFM...
RFM ready: (1035957, 65)
RAM: 26.2%

Training RNN on RFM features...
Epoch 1/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 23s 10ms/step - AUC: 0.8464 - loss: 0.4799 - val_AUC: 0.9075 - val_loss: 0.2444
Epoch 2/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - AUC: 0.9112 - loss: 0.3754 - val_AUC: 0.9220 - val_loss: 0.1933
Epoch 3/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - AUC: 0.9210 - loss: 0.3559 - val_AUC: 0.9244 - val_loss: 0.1909
Epoch 4/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - AUC: 0.9253 - loss: 0.3466 - val_AUC: 0.9235 - val_loss: 0.1840
Epoch 5/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - AUC: 0.9277 - loss: 0.3414 - val_AUC: 0.9286 - val_loss: 0.1719
Epoch 6/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - AUC: 0.9279 - loss: 0.3405 - val_AUC: 0.9329 - val_loss: 0.1717
Epoch 7/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - AUC: 0.9314 - loss: 0.3327 - val_AUC: 0.9337 - val_loss: 0.1906
Epoch 8/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 21s 10ms/step - AUC

# Train RNN on HOBA Features

This cell trains an LSTM on the HOBA feature vector.

Unlike the original paper, which represents each transaction as a
(feature types × recent transactions) matrix, we use the same adaptation
as our CNN implementation by treating the 891 engineered HOBA features
as a one-dimensional sequence.

In [ ]:
# ============================================================
# Load HOBA
# ============================================================

print("\nLoading HOBA...")

train_hoba = pd.read_parquet(f"{DRIVE_PATH}/train_hoba_scaled.parquet")
test_hoba  = pd.read_parquet(f"{DRIVE_PATH}/test_hoba_scaled.parquet")

hoba_cols = [c for c in train_hoba.columns if c.startswith("hoba_")]

X_train = train_hoba[hoba_cols].values.astype(np.float32)
X_test  = test_hoba[hoba_cols].values.astype(np.float32)

y_train = train_hoba["is_fraud"].astype(int).values
y_test  = test_hoba["is_fraud"].astype(int).values

del train_hoba, test_hoba
gc.collect()

print(f"HOBA ready: {X_train.shape}")
print(f"RAM: {psutil.virtual_memory().percent}%")

# ============================================================
# Class Weights
# ============================================================

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1]),
    y=y_train
)

class_weight_dict = {
    0: float(weights[0]),
    1: float(weights[1])
}

# ============================================================
# Build RNN
# ============================================================

model = Sequential([

    Reshape((len(hoba_cols), 1), input_shape=(len(hoba_cols),)),

    LSTM(
        64,
        return_sequences=False
    ),

    Dropout(0.30),

    Dense(
        32,
        activation="relu"
    ),

    Dropout(0.20),

    Dense(
        1,
        activation="sigmoid"
    )

])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["AUC"]
)

early = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("\nTraining RNN on HOBA features...")

history = model.fit(

    X_train,
    y_train,

    epochs=20,

    batch_size=512,

    validation_split=0.20,

    class_weight=class_weight_dict,

    callbacks=[early],

    verbose=1

)

# ============================================================
# Evaluate
# ============================================================

y_prob = model.predict(
    X_test,
    batch_size=4096,
    verbose=1
).ravel()

y_pred = (y_prob >= 0.5).astype(int)

evaluate_model(
    "RNN — HOBA",
    y_test,
    y_pred,
    y_prob
)

del X_train
del X_test
del y_train
del y_test
gc.collect()

print(f"RAM after RNN-HOBA: {psutil.virtual_memory().percent}%")


Loading HOBA...


In [ ]:
# ============================================================
# Load HOBA
# ============================================================

print("\nLoading HOBA...")

train_hoba = pd.read_parquet(f"{DRIVE_PATH}/train_hoba_scaled.parquet")
test_hoba  = pd.read_parquet(f"{DRIVE_PATH}/test_hoba_scaled.parquet")

hoba_cols = [c for c in train_hoba.columns if c.startswith("hoba_")]

print(f"Number of HOBA features: {len(hoba_cols)}")

# Safety check
assert len(hoba_cols) == 891, \
    f"Expected 891 HOBA features, found {len(hoba_cols)}."

X_train = train_hoba[hoba_cols].to_numpy(dtype=np.float32)
X_test  = test_hoba[hoba_cols].to_numpy(dtype=np.float32)

y_train = train_hoba["is_fraud"].astype(np.int8).to_numpy()
y_test  = test_hoba["is_fraud"].astype(np.int8).to_numpy()

del train_hoba, test_hoba
gc.collect()

print(f"HOBA ready: {X_train.shape}")
print(f"RAM: {psutil.virtual_memory().percent}%")

# ============================================================
# Class Weights
# ============================================================

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)

class_weight_dict = {
    0: float(weights[0]),
    1: float(weights[1])
}

# ============================================================
# Build RNN
# ============================================================

model = Sequential([

    Reshape((99, 9), input_shape=(891,)),

    LSTM(
        64,
        return_sequences=False
    ),

    Dropout(0.30),

    Dense(
        32,
        activation="relu"
    ),

    Dropout(0.20),

    Dense(
        1,
        activation="sigmoid"
    )

])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["AUC"]
)

early = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("\nTraining RNN on HOBA features...")

history = model.fit(

    X_train,
    y_train,

    epochs=20,

    batch_size=128,

    validation_split=0.20,

    class_weight=class_weight_dict,

    callbacks=[early],

    verbose=1

)

# ============================================================
# Evaluate
# ============================================================

y_prob = model.predict(
    X_test,
    batch_size=2048,
    verbose=1
).ravel()

y_pred = (y_prob >= 0.5).astype(int)

evaluate_model(
    "RNN — HOBA",
    y_test,
    y_pred,
    y_prob
)

del X_train
del X_test
del y_train
del y_test
gc.collect()

print(f"RAM after RNN-HOBA: {psutil.virtual_memory().percent}%")


Loading HOBA...
Number of HOBA features: 891


In [ ]:
# ============================================================
# Load HOBA (Memory Efficient)
# ============================================================

print("\nLoading HOBA...")

pf_train = pq.ParquetFile(f"{DRIVE_PATH}/train_hoba_scaled.parquet")

all_cols = pf_train.schema_arrow.names
hoba_cols = [c for c in all_cols if c.startswith("hoba_")]

print(f"Number of HOBA features: {len(hoba_cols)}")

assert len(hoba_cols) == 891

n_rows_train = pf_train.metadata.num_rows
n_features = len(hoba_cols)

print(f"Allocating train array ({n_rows_train:,} × {n_features})")

X_train = np.empty((n_rows_train, n_features), dtype=np.float32)
y_train = np.empty(n_rows_train, dtype=np.int8)

offset = 0

for batch in pf_train.iter_batches(
        batch_size=100000,
        columns=hoba_cols + ["is_fraud"]):

    n = batch.num_rows

    for i, col in enumerate(hoba_cols):
        X_train[offset:offset+n, i] = \
            batch.column(col).to_numpy(zero_copy_only=False)

    y_train[offset:offset+n] = \
        batch.column("is_fraud").to_numpy(zero_copy_only=False)

    offset += n

    del batch
    gc.collect()

print(f"Train loaded. RAM: {psutil.virtual_memory().percent}%")

# ============================================================
# Load Test
# ============================================================

pf_test = pq.ParquetFile(f"{DRIVE_PATH}/test_hoba_scaled.parquet")

n_rows_test = pf_test.metadata.num_rows

print(f"Allocating test array ({n_rows_test:,} × {n_features})")

X_test = np.empty((n_rows_test, n_features), dtype=np.float32)
y_test = np.empty(n_rows_test, dtype=np.int8)

offset = 0

for batch in pf_test.iter_batches(
        batch_size=100000,
        columns=hoba_cols + ["is_fraud"]):

    n = batch.num_rows

    for i, col in enumerate(hoba_cols):
        X_test[offset:offset+n, i] = \
            batch.column(col).to_numpy(zero_copy_only=False)

    y_test[offset:offset+n] = \
        batch.column("is_fraud").to_numpy(zero_copy_only=False)

    offset += n

    del batch
    gc.collect()

print(f"Test loaded. RAM: {psutil.virtual_memory().percent}%")

# ============================================================
# Class Weights
# ============================================================

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)

class_weight_dict = {
    0: float(weights[0]),
    1: float(weights[1])
}

# ============================================================
# Build RNN
# ============================================================

model = Sequential([

    Reshape((99, 9), input_shape=(891,)),

    LSTM(
        64,
        return_sequences=False
    ),

    Dropout(0.30),

    Dense(
        32,
        activation="relu"
    ),

    Dropout(0.20),

    Dense(
        1,
        activation="sigmoid"
    )

])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["AUC"]
)

early = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("\nModel built successfully.")
print(model.summary())


Loading HOBA...
Number of HOBA features: 891
Allocating train array (1,035,957 × 891)
Train loaded. RAM: 56.4%
Allocating test array (346,537 × 891)
Test loaded. RAM: 68.3%

Model built successfully.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape (Reshape)               │ (None, 99, 9)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,057 (82.25 KB)

 Trainable params: 21,057 (82.25 KB)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
# ============================================================
# SECTION 14 — RNN — FULLY SELF-CONTAINED
# ============================================================

import gc
import os
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    roc_curve
)

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    LSTM,
    Reshape
)

from tensorflow.keras.callbacks import EarlyStopping

# ============================================================
# Global Settings
# ============================================================

DRIVE_PATH = "/content/drive/MyDrive/HOBA_Fraud_Detection_V2"

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("GPU:", tf.config.list_physical_devices("GPU"))

# ============================================================
# Evaluation Helpers
# ============================================================

def get_metrics_at_fpr(y_true, y_prob, fpr_threshold):

    fpr, tpr, thresholds = roc_curve(y_true, y_prob)

    valid = np.where(fpr <= fpr_threshold)[0]

    if len(valid) == 0:
        return {
            "f1":0,
            "precision":0,
            "recall":0,
            "accuracy":0
        }

    best = valid[np.argmax(tpr[valid])]

    threshold = thresholds[best]

    y_pred = (y_prob >= threshold).astype(int)

    TP = ((y_pred==1)&(y_true==1)).sum()
    TN = ((y_pred==0)&(y_true==0)).sum()
    FP = ((y_pred==1)&(y_true==0)).sum()
    FN = ((y_pred==0)&(y_true==1)).sum()

    precision = TP/(TP+FP) if TP+FP else 0
    recall    = TP/(TP+FN) if TP+FN else 0

    f1 = (
        2*precision*recall/(precision+recall)
        if precision+recall else 0
    )

    accuracy = (TP+TN)/len(y_true)

    return {
        "f1":f1,
        "precision":precision,
        "recall":recall,
        "accuracy":accuracy
    }


def evaluate_model(name, y_true, y_pred, y_prob):

    auc = roc_auc_score(y_true, y_prob)

    f1 = f1_score(y_true, y_pred)

    precision = precision_score(y_true, y_pred)

    recall = recall_score(y_true, y_pred)

    accuracy = (y_true==y_pred).mean()

    m1 = get_metrics_at_fpr(y_true, y_prob,0.01)
    m3 = get_metrics_at_fpr(y_true, y_prob,0.03)

    print(f"\n=== {name} ===")

    print(
        f"Overall  — "
        f"AUC:{auc:.4f} "
        f"F1:{f1:.4f} "
        f"Pre:{precision:.4f} "
        f"Rec:{recall:.4f} "
        f"Acc:{accuracy:.4f}"
    )

    print(
        f"@1%FPR — "
        f"F1:{m1['f1']:.4f} "
        f"Pre:{m1['precision']:.4f} "
        f"Rec:{m1['recall']:.4f}"
    )

    print(
        f"@3%FPR — "
        f"F1:{m3['f1']:.4f} "
        f"Pre:{m3['precision']:.4f} "
        f"Rec:{m3['recall']:.4f}"
    )

    result = {

        "name":name,

        "auc":auc,

        "f1":f1,

        "precision":precision,

        "recall":recall,

        "accuracy":accuracy,

        "f1_1fpr":m1["f1"],
        "precision_1fpr":m1["precision"],
        "recall_1fpr":m1["recall"],
        "accuracy_1fpr":m1["accuracy"],

        "f1_3fpr":m3["f1"],
        "precision_3fpr":m3["precision"],
        "recall_3fpr":m3["recall"],
        "accuracy_3fpr":m3["accuracy"]

    }

    path = f"{DRIVE_PATH}/results.json"

    if os.path.exists(path):
        results = json.load(open(path))
    else:
        results = []

    results = [r for r in results if r["name"] != name]

    results.append(result)

    json.dump(results, open(path,"w"), indent=2)

    print("Result saved.")

    return result


early_stop = EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True,

    verbose=1

)

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# ============================================================
# HOBA STREAMING SETUP
# ============================================================

print("\nSetting up HOBA streaming...")

pf_train = pq.ParquetFile(
    f"{DRIVE_PATH}/train_hoba_scaled.parquet"
)

hoba_cols = [
    c
    for c in pf_train.schema_arrow.names
    if c.startswith("hoba_")
]

n_cols = len(hoba_cols)

n_rows_train = pf_train.metadata.num_rows

pf_test = pq.ParquetFile(
    f"{DRIVE_PATH}/test_hoba_scaled.parquet"
)

n_rows_test = pf_test.metadata.num_rows

VAL_FRACTION = 0.10

n_val = int(n_rows_train * VAL_FRACTION)

n_train = n_rows_train - n_val

print(f"HOBA features : {n_cols}")
print(f"Training rows : {n_train:,}")
print(f"Validation rows : {n_val:,}")
print(f"Test rows : {n_rows_test:,}")

# ------------------------------------------------------------
# Class weights
# ------------------------------------------------------------

labels = np.concatenate([

    batch.column("is_fraud").to_numpy(
        zero_copy_only=False
    )

    for batch in pf_train.iter_batches(

        batch_size=200000,

        columns=["is_fraud"]

    )

])

weights = compute_class_weight(

    class_weight="balanced",

    classes=np.array([0,1]),

    y=labels

)

class_weight_dict = {

    0: float(weights[0]),

    1: float(weights[1])

}

del labels

gc.collect()

print(f"RAM : {psutil.virtual_memory().percent}%")

# ------------------------------------------------------------
# Dataset Generator
# ------------------------------------------------------------

BATCH_SIZE = 512

train_path = f"{DRIVE_PATH}/train_hoba_scaled.parquet"

test_path = f"{DRIVE_PATH}/test_hoba_scaled.parquet"

def make_generator(

    parquet_path,

    row_start,

    row_end,

    batch_size,

    cols

):

    def gen():

        pf = pq.ParquetFile(parquet_path)

        row_cursor = 0

        for batch in pf.iter_batches(

            batch_size=batch_size,

            columns=cols + ["is_fraud"]

        ):

            n = batch.num_rows

            lo = max(row_start, row_cursor)

            hi = min(row_end, row_cursor + n)

            row_cursor += n

            if lo >= hi:

                continue

            local_lo = lo - (row_cursor - n)

            local_hi = hi - (row_cursor - n)

            X = np.column_stack([

                batch.column(c).to_numpy(
                    zero_copy_only=False
                )[local_lo:local_hi]

                for c in cols

            ]).astype(np.float32)

            y = batch.column("is_fraud").to_numpy(
                zero_copy_only=False
            )[local_lo:local_hi].astype(np.int8)

            yield X, y

    return gen

signature = (

    tf.TensorSpec(

        shape=(None, n_cols),

        dtype=tf.float32

    ),

    tf.TensorSpec(

        shape=(None,),

        dtype=tf.int8

    )

)

train_ds = tf.data.Dataset.from_generator(

    make_generator(

        train_path,

        0,

        n_train,

        BATCH_SIZE,

        hoba_cols

    ),

    output_signature=signature

).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(

    make_generator(

        train_path,

        n_train,

        n_rows_train,

        BATCH_SIZE,

        hoba_cols

    ),

    output_signature=signature

).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_generator(

    make_generator(

        test_path,

        0,

        n_rows_test,

        BATCH_SIZE,

        hoba_cols

    ),

    output_signature=signature

).prefetch(tf.data.AUTOTUNE)

print("Streaming datasets ready.")


Setting up HOBA streaming...
HOBA features : 891
Training rows : 932,362
Validation rows : 103,595
Test rows : 346,537
RAM : 16.0%
Streaming datasets ready.


In [ ]:
# ============================================================
# Build RNN
# ============================================================

train_ds_rnn = train_ds.map(
    lambda x, y: (tf.reshape(x, (-1, 99, 9)), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds_rnn = val_ds.map(
    lambda x, y: (tf.reshape(x, (-1, 99, 9)), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

test_ds_rnn = test_ds.map(
    lambda x, y: (tf.reshape(x, (-1, 99, 9)), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

# ============================================================

def build_rnn():

    model = Sequential([

        LSTM(
            32,
            input_shape=(99,9)
        ),

        Dropout(0.30),

        Dense(
            32,
            activation="relu"
        ),

        Dropout(0.20),

        Dense(
            1,
            activation="sigmoid"
        )

    ])

    model.compile(

        optimizer="adam",

        loss="binary_crossentropy",

        metrics=["AUC"]

    )

    return model


print("\nTraining RNN on HOBA features...")

rnn_hoba = build_rnn()

history = rnn_hoba.fit(

    train_ds_rnn,

    validation_data=val_ds_rnn,

    epochs=20,

    class_weight=class_weight_dict,

    callbacks=[early_stop],

    verbose=1

)

print(f"RAM after fit: {psutil.virtual_memory().percent}%")

# ============================================================
# Prediction
# ============================================================

print("\nPredicting...")

y_prob_list = []

y_test_list = []

for X_batch, y_batch in test_ds_rnn:

    y_prob_list.append(

        rnn_hoba.predict(
            X_batch,
            verbose=0
        ).flatten()

    )

    y_test_list.append(
        y_batch.numpy()
    )

y_prob = np.concatenate(y_prob_list)

y_test = np.concatenate(y_test_list)

y_pred = (y_prob >= 0.5).astype(int)

evaluate_model(

    "RNN — HOBA",

    y_test,

    y_pred,

    y_prob

)

print(f"RAM after evaluation: {psutil.virtual_memory().percent}%")

print_results_table()


Training RNN on HOBA features...
Epoch 1/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 101s 51ms/step - AUC: 0.8261 - loss: 0.5122 - val_AUC: 0.8522 - val_loss: 0.3658
Epoch 2/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 91s 49ms/step - AUC: 0.8541 - loss: 0.4694 - val_AUC: 0.8747 - val_loss: 0.2246
Epoch 3/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 88s 48ms/step - AUC: 0.9066 - loss: 0.3843 - val_AUC: 0.9000 - val_loss: 0.0987
Epoch 4/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 142s 78ms/step - AUC: 0.9376 - loss: 0.3171 - val_AUC: 0.9582 - val_loss: 0.1285
Epoch 5/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 142s 78ms/step - AUC: 0.9527 - loss: 0.2755 - val_AUC: 0.9660 - val_loss: 0.2072
Epoch 6/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 143s 78ms/step - AUC: 0.9600 - loss: 0.2511 - val_AUC: 0.9720 - val_loss: 0.1734
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.
RAM after fit: 25.3%

Predicting...

=== RNN — HOBA ===
Overall  — AUC:0.9498 F1:0.1479 Pre:0.0839 Rec:0.6215 Acc:0.9569
@1%FPR — F1:0.2524 Pre:0.1

In [ ]:
import json

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
results_path = f'{DRIVE_PATH}/results.json'
all_results = json.load(open(results_path))

print(f"{'Model':<25} {'AUC':>6} {'F1':>6} {'Rec':>6} {'Acc':>6} | {'F1@1%':>6} {'Pre@1%':>7} {'Rec@1%':>7} | {'F1@3%':>6} {'Rec@3%':>7}")
print("-"*105)
for r in all_results:
    print(f"{r['name']:<25} {r['auc']:>6.4f} {r['f1']:>6.4f} {r['recall']:>6.4f} {r.get('accuracy',0):>6.4f} | "
          f"{r.get('f1_1fpr',0):>6.4f} {r.get('precision_1fpr',0):>7.4f} {r.get('recall_1fpr',0):>7.4f} | "
          f"{r.get('f1_3fpr',0):>6.4f} {r.get('recall_3fpr',0):>7.4f}")

Model                        AUC     F1    Rec    Acc |  F1@1%  Pre@1%  Rec@1% |  F1@3%  Rec@3%
---------------------------------------------------------------------------------------------------------
Random Forest — RFM       0.9513 0.2881 0.7139 0.0000 | 0.0000  0.0000  0.6349 | 0.0000  0.7604
Random Forest — HOBA      0.9933 0.3355 0.9291 0.0000 | 0.0000  0.0000  0.8773 | 0.0000  0.9459
SVM — RFM                 0.8849 0.0688 0.7398 0.0000 | 0.0000  0.0000  0.3426 | 0.0000  0.4969
SVM — HOBA                0.9597 0.1548 0.8860 0.0000 | 0.0000  0.0000  0.6004 | 0.0000  0.8150
BPNN — RFM                0.9516 0.1572 0.8131 0.0000 | 0.0000  0.0000  0.6814 | 0.0000  0.7638
BPNN — HOBA               0.9932 0.2799 0.9363 0.0000 | 0.0000  0.0000  0.8591 | 0.0000  0.9401
DBN — RFM                 0.9501 0.1494 0.8179 0.9439 | 0.3999  0.2876  0.6564 | 0.2251  0.7547
DBN — HOBA                0.9919 0.2761 0.9315 0.9706 | 0.4911  0.3436  0.8606 | 0.2716  0.9324
CNN — RFM                 0.95

## Section 14: Rerun RF, SVM, BPNN with Complete Metrics
Retraining RF, SVM, and BPNN to compute full metric set matching
paper's Table 3 and Table 5a: Overall + constrained FPR at 1% and 3%.
Overwrites partial results in results.json with complete metrics.

In [ ]:
# ============================================================
# SECTION 14 — RERUN RF + SVM — FULLY SELF-CONTAINED
# ============================================================
import gc, os, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve)

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Evaluation helpers ────────────────────────────────────────
def get_metrics_at_fpr(y_true, y_prob, fpr_threshold):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    valid_idx = np.where(fpr <= fpr_threshold)[0]
    if len(valid_idx) == 0:
        return {'f1': 0, 'precision': 0, 'recall': 0, 'accuracy': 0}
    best_idx = valid_idx[np.argmax(tpr[valid_idx])]
    thresh = thresholds[best_idx]
    y_pred_thresh = (y_prob >= thresh).astype(int)
    TP = ((y_pred_thresh==1)&(y_true==1)).sum()
    TN = ((y_pred_thresh==0)&(y_true==0)).sum()
    FP = ((y_pred_thresh==1)&(y_true==0)).sum()
    FN = ((y_pred_thresh==0)&(y_true==1)).sum()
    precision = TP/(TP+FP) if (TP+FP)>0 else 0
    recall    = TP/(TP+FN) if (TP+FN)>0 else 0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
    return {'f1': f1, 'precision': precision, 'recall': recall,
            'accuracy': (TP+TN)/len(y_true)}

def evaluate_model(name, y_true, y_pred, y_prob):
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    acc  = (y_pred==y_true).sum()/len(y_true)
    m1   = get_metrics_at_fpr(y_true, y_prob, 0.01)
    m3   = get_metrics_at_fpr(y_true, y_prob, 0.03)
    print(f"\n=== {name} ===")
    print(f"Overall  — AUC:{auc:.4f} F1:{f1:.4f} Pre:{prec:.4f} Rec:{rec:.4f} Acc:{acc:.4f}")
    print(f"@ 1% FPR — F1:{m1['f1']:.4f} Pre:{m1['precision']:.4f} Rec:{m1['recall']:.4f} Acc:{m1['accuracy']:.4f}")
    print(f"@ 3% FPR — F1:{m3['f1']:.4f} Pre:{m3['precision']:.4f} Rec:{m3['recall']:.4f} Acc:{m3['accuracy']:.4f}")
    result = {
        'name': name, 'auc': auc, 'f1': f1, 'precision': prec,
        'recall': rec, 'accuracy': acc,
        'f1_1fpr': m1['f1'], 'precision_1fpr': m1['precision'],
        'recall_1fpr': m1['recall'], 'accuracy_1fpr': m1['accuracy'],
        'f1_3fpr': m3['f1'], 'precision_3fpr': m3['precision'],
        'recall_3fpr': m3['recall'], 'accuracy_3fpr': m3['accuracy']
    }
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    all_results = [r for r in all_results if r['name'] != name]
    all_results.append(result)
    json.dump(all_results, open(results_path, 'w'), indent=2)
    print("Result saved to Drive.")
    return result

def print_results_table():
    results_path = f'{DRIVE_PATH}/results.json'
    if not os.path.exists(results_path):
        print("No results saved yet.")
        return
    all_results = json.load(open(results_path))
    print(f"\n{'Model':<25} {'AUC':>6} {'F1':>6} {'Rec':>6} {'Acc':>6} | {'F1@1%':>6} {'Pre@1%':>7} {'Rec@1%':>7} | {'F1@3%':>6} {'Rec@3%':>7}")
    print("-"*105)
    for r in all_results:
        print(f"{r['name']:<25} {r['auc']:>6.4f} {r['f1']:>6.4f} {r['recall']:>6.4f} {r.get('accuracy',0):>6.4f} | "
              f"{r.get('f1_1fpr',0):>6.4f} {r.get('precision_1fpr',0):>7.4f} {r.get('recall_1fpr',0):>7.4f} | "
              f"{r.get('f1_3fpr',0):>6.4f} {r.get('recall_3fpr',0):>7.4f}")

# ── Load RFM ─────────────────────────────────────────────────
print("Loading RFM...")
train_rfm_df = pd.read_parquet(f'{DRIVE_PATH}/train_rfm.parquet')
test_rfm_df  = pd.read_parquet(f'{DRIVE_PATH}/test_rfm.parquet')
rfm_cols     = [c for c in train_rfm_df.columns if c.startswith('rfm_')]
y_train      = train_rfm_df['is_fraud'].astype(int).to_numpy()
y_test       = test_rfm_df['is_fraud'].astype(int).to_numpy()
X_train_rfm_raw = train_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
X_test_rfm_raw  = test_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
del train_rfm_df, test_rfm_df; gc.collect()

scaler_rfm  = StandardScaler()
X_train_rfm = scaler_rfm.fit_transform(X_train_rfm_raw)
X_test_rfm  = scaler_rfm.transform(X_test_rfm_raw)
del X_train_rfm_raw, X_test_rfm_raw; gc.collect()
print(f"RFM ready: {X_train_rfm.shape} | RAM: {psutil.virtual_memory().percent}%")

# ── Load HOBA arrays ──────────────────────────────────────────
print("\nLoading HOBA train...")
pf = pq.ParquetFile(f'{DRIVE_PATH}/train_hoba_scaled.parquet')
hoba_cols = [c for c in pf.schema_arrow.names if c.startswith('hoba_')]
n_cols = len(hoba_cols)
n_rows = pf.metadata.num_rows

X_train_hoba = np.empty((n_rows, n_cols), dtype=np.float32)
row_offset = 0
for batch in pf.iter_batches(batch_size=100_000, columns=hoba_cols):
    n = batch.num_rows
    for i, c in enumerate(hoba_cols):
        X_train_hoba[row_offset:row_offset+n, i] = batch.column(c).to_numpy(zero_copy_only=False)
    row_offset += n
    del batch; gc.collect()
print(f"HOBA train ready: {X_train_hoba.shape} | RAM: {psutil.virtual_memory().percent}%")

print("Loading HOBA test...")
pf_test = pq.ParquetFile(f'{DRIVE_PATH}/test_hoba_scaled.parquet')
n_rows_test = pf_test.metadata.num_rows
X_test_hoba = np.empty((n_rows_test, n_cols), dtype=np.float32)
y_test_hoba = np.empty(n_rows_test, dtype=np.int8)
row_offset = 0
for batch in pf_test.iter_batches(batch_size=100_000, columns=hoba_cols + ['is_fraud']):
    n = batch.num_rows
    for i, c in enumerate(hoba_cols):
        X_test_hoba[row_offset:row_offset+n, i] = batch.column(c).to_numpy(zero_copy_only=False)
    y_test_hoba[row_offset:row_offset+n] = batch.column('is_fraud').to_numpy(zero_copy_only=False)
    row_offset += n
    del batch; gc.collect()
print(f"HOBA test ready: {X_test_hoba.shape} | RAM: {psutil.virtual_memory().percent}%")

# ── Random Forest ─────────────────────────────────────────────
print("\nTraining RF on RFM...")
rf_rfm = RandomForestClassifier(n_estimators=100, max_depth=10,
                                 class_weight='balanced',
                                 random_state=RANDOM_SEED, n_jobs=-1)
rf_rfm.fit(X_train_rfm, y_train)
y_pred = rf_rfm.predict(X_test_rfm)
y_prob = rf_rfm.predict_proba(X_test_rfm)[:, 1]
evaluate_model("Random Forest — RFM", y_test, y_pred, y_prob)
del rf_rfm, y_pred, y_prob; gc.collect()

print("\nTraining RF on HOBA...")
rf_hoba = RandomForestClassifier(n_estimators=100, max_depth=10,
                                  class_weight='balanced',
                                  random_state=RANDOM_SEED, n_jobs=-1)
rf_hoba.fit(X_train_hoba, y_train)
y_pred = rf_hoba.predict(X_test_hoba)
y_prob = rf_hoba.predict_proba(X_test_hoba)[:, 1]
evaluate_model("Random Forest — HOBA", y_test_hoba, y_pred, y_prob)
del rf_hoba, y_pred, y_prob; gc.collect()
print(f"RAM after RF: {psutil.virtual_memory().percent}%")

# ── SVM ───────────────────────────────────────────────────────
print("\nTraining SVM on RFM...")
svm_rfm = SGDClassifier(loss='hinge', class_weight='balanced',
                         random_state=RANDOM_SEED, max_iter=1000, n_jobs=-1)
svm_rfm.fit(X_train_rfm, y_train)
y_pred = svm_rfm.predict(X_test_rfm)
y_prob = svm_rfm.decision_function(X_test_rfm)
evaluate_model("SVM — RFM", y_test, y_pred, y_prob)
del svm_rfm, y_pred, y_prob; gc.collect()

print("\nTraining SVM on HOBA...")
svm_hoba = SGDClassifier(loss='hinge', class_weight='balanced',
                          random_state=RANDOM_SEED, max_iter=1000, n_jobs=-1)
svm_hoba.fit(X_train_hoba, y_train)
y_pred = svm_hoba.predict(X_test_hoba)
y_prob = svm_hoba.decision_function(X_test_hoba)
evaluate_model("SVM — HOBA", y_test_hoba, y_pred, y_prob)
del svm_hoba, y_pred, y_prob; gc.collect()
print(f"RAM after SVM: {psutil.virtual_memory().percent}%")

print("\n")
print_results_table()

Loading RFM...
RFM ready: (1035957, 65) | RAM: 32.4%

Loading HOBA train...
HOBA train ready: (1035957, 891) | RAM: 67.6%
Loading HOBA test...
HOBA test ready: (346537, 891) | RAM: 76.8%

Training RF on RFM...

=== Random Forest — RFM ===
Overall  — AUC:0.9513 F1:0.2881 Pre:0.1805 Rec:0.7139 Acc:0.9788
@ 1% FPR — F1:0.3898 Pre:0.2812 Rec:0.6349 Acc:0.9880
@ 3% FPR — F1:0.2272 Pre:0.1335 Rec:0.7604 Acc:0.9688
Result saved to Drive.

Training RF on HOBA...

=== Random Forest — HOBA ===
Overall  — AUC:0.9933 F1:0.3355 Pre:0.2047 Rec:0.9291 Acc:0.9778
@ 1% FPR — F1:0.4981 Pre:0.3478 Rec:0.8773 Acc:0.9894
@ 3% FPR — F1:0.2762 Pre:0.1617 Rec:0.9459 Acc:0.9701
Result saved to Drive.
RAM after RF: 72.4%

Training SVM on RFM...

=== SVM — RFM ===
Overall  — AUC:0.8849 F1:0.0688 Pre:0.0361 Rec:0.7398 Acc:0.8793
@ 1% FPR — F1:0.2294 Pre:0.1724 Rec:0.3426 Acc:0.9861
@ 3% FPR — F1:0.1542 Pre:0.0912 Rec:0.4969 Acc:0.9672
Result saved to Drive.

Training SVM on HOBA...

=== SVM — HOBA ===
Overall  — 

In [2]:
# ============================================================
# SECTION 14b — RERUN BPNN — FULLY SELF-CONTAINED
# ============================================================
import gc, os, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Evaluation helpers ────────────────────────────────────────
def get_metrics_at_fpr(y_true, y_prob, fpr_threshold):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    valid_idx = np.where(fpr <= fpr_threshold)[0]
    if len(valid_idx) == 0:
        return {'f1': 0, 'precision': 0, 'recall': 0, 'accuracy': 0}
    best_idx = valid_idx[np.argmax(tpr[valid_idx])]
    thresh = thresholds[best_idx]
    y_pred_thresh = (y_prob >= thresh).astype(int)
    TP = ((y_pred_thresh==1)&(y_true==1)).sum()
    TN = ((y_pred_thresh==0)&(y_true==0)).sum()
    FP = ((y_pred_thresh==1)&(y_true==0)).sum()
    FN = ((y_pred_thresh==0)&(y_true==1)).sum()
    precision = TP/(TP+FP) if (TP+FP)>0 else 0
    recall    = TP/(TP+FN) if (TP+FN)>0 else 0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
    return {'f1': f1, 'precision': precision, 'recall': recall,
            'accuracy': (TP+TN)/len(y_true)}

def evaluate_model(name, y_true, y_pred, y_prob):
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    acc  = (y_pred==y_true).sum()/len(y_true)
    m1   = get_metrics_at_fpr(y_true, y_prob, 0.01)
    m3   = get_metrics_at_fpr(y_true, y_prob, 0.03)
    print(f"\n=== {name} ===")
    print(f"Overall  — AUC:{auc:.4f} F1:{f1:.4f} Pre:{prec:.4f} Rec:{rec:.4f} Acc:{acc:.4f}")
    print(f"@ 1% FPR — F1:{m1['f1']:.4f} Pre:{m1['precision']:.4f} Rec:{m1['recall']:.4f} Acc:{m1['accuracy']:.4f}")
    print(f"@ 3% FPR — F1:{m3['f1']:.4f} Pre:{m3['precision']:.4f} Rec:{m3['recall']:.4f} Acc:{m3['accuracy']:.4f}")
    result = {
        'name': name, 'auc': auc, 'f1': f1, 'precision': prec,
        'recall': rec, 'accuracy': acc,
        'f1_1fpr': m1['f1'], 'precision_1fpr': m1['precision'],
        'recall_1fpr': m1['recall'], 'accuracy_1fpr': m1['accuracy'],
        'f1_3fpr': m3['f1'], 'precision_3fpr': m3['precision'],
        'recall_3fpr': m3['recall'], 'accuracy_3fpr': m3['accuracy']
    }
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    all_results = [r for r in all_results if r['name'] != name]
    all_results.append(result)
    json.dump(all_results, open(results_path, 'w'), indent=2)
    print("Result saved to Drive.")
    return result

def print_results_table():
    results_path = f'{DRIVE_PATH}/results.json'
    if not os.path.exists(results_path):
        print("No results saved yet.")
        return
    all_results = json.load(open(results_path))
    print(f"\n{'Model':<25} {'AUC':>6} {'F1':>6} {'Rec':>6} {'Acc':>6} | {'F1@1%':>6} {'Pre@1%':>7} {'Rec@1%':>7} | {'F1@3%':>6} {'Rec@3%':>7}")
    print("-"*105)
    for r in all_results:
        print(f"{r['name']:<25} {r['auc']:>6.4f} {r['f1']:>6.4f} {r['recall']:>6.4f} {r.get('accuracy',0):>6.4f} | "
              f"{r.get('f1_1fpr',0):>6.4f} {r.get('precision_1fpr',0):>7.4f} {r.get('recall_1fpr',0):>7.4f} | "
              f"{r.get('f1_3fpr',0):>6.4f} {r.get('recall_3fpr',0):>7.4f}")

# ── Load RFM ─────────────────────────────────────────────────
print("Loading RFM...")
train_rfm_df = pd.read_parquet(f'{DRIVE_PATH}/train_rfm.parquet')
test_rfm_df  = pd.read_parquet(f'{DRIVE_PATH}/test_rfm.parquet')
rfm_cols     = [c for c in train_rfm_df.columns if c.startswith('rfm_')]
y_train_rfm  = train_rfm_df['is_fraud'].astype(int).to_numpy()
y_test_rfm   = test_rfm_df['is_fraud'].astype(int).to_numpy()
X_train_rfm_raw = train_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
X_test_rfm_raw  = test_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
del train_rfm_df, test_rfm_df; gc.collect()

scaler_rfm  = StandardScaler()
X_train_rfm = scaler_rfm.fit_transform(X_train_rfm_raw)
X_test_rfm  = scaler_rfm.transform(X_test_rfm_raw)
del X_train_rfm_raw, X_test_rfm_raw; gc.collect()
print(f"RFM ready: {X_train_rfm.shape} | RAM: {psutil.virtual_memory().percent}%")

weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train_rfm)
class_weight_dict = {0: float(weights[0]), 1: float(weights[1])}
early_stop = EarlyStopping(monitor='val_loss', patience=3,
                            restore_best_weights=True, verbose=1)

def build_bpnn(input_dim):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

# ── BPNN on RFM ──────────────────────────────────────────────
print("\nTraining BPNN on RFM features...")
bpnn_rfm = build_bpnn(input_dim=X_train_rfm.shape[1])
bpnn_rfm.fit(X_train_rfm, y_train_rfm, epochs=20, batch_size=512,
             validation_split=0.1, class_weight=class_weight_dict,
             callbacks=[early_stop], verbose=1)
y_prob = bpnn_rfm.predict(X_test_rfm, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)
evaluate_model("BPNN — RFM", y_test_rfm, y_pred, y_prob)
del bpnn_rfm, y_pred, y_prob, X_train_rfm, X_test_rfm; gc.collect()
print(f"RAM after BPNN-RFM: {psutil.virtual_memory().percent}%")

# ── HOBA streaming setup ──────────────────────────────────────
print("\nSetting up HOBA streaming...")
pf_train = pq.ParquetFile(f'{DRIVE_PATH}/train_hoba_scaled.parquet')
hoba_cols = [c for c in pf_train.schema_arrow.names if c.startswith('hoba_')]
n_cols = len(hoba_cols)
n_rows_train = pf_train.metadata.num_rows
pf_test_f = pq.ParquetFile(f'{DRIVE_PATH}/test_hoba_scaled.parquet')
n_rows_test = pf_test_f.metadata.num_rows
n_val   = int(n_rows_train * 0.1)
n_train = n_rows_train - n_val

labels = np.concatenate([
    b.column('is_fraud').to_numpy(zero_copy_only=False)
    for b in pf_train.iter_batches(batch_size=200_000, columns=['is_fraud'])
])
wh = compute_class_weight('balanced', classes=np.array([0,1]), y=labels)
cw_hoba = {0: float(wh[0]), 1: float(wh[1])}
del labels; gc.collect()
print(f"RAM: {psutil.virtual_memory().percent}%")

BATCH_SIZE = 512
train_path = f'{DRIVE_PATH}/train_hoba_scaled.parquet'
test_path  = f'{DRIVE_PATH}/test_hoba_scaled.parquet'

def make_generator(path, row_start, row_end, bsz, cols):
    def gen():
        pf = pq.ParquetFile(path)
        rc = 0
        for batch in pf.iter_batches(batch_size=bsz, columns=cols+['is_fraud']):
            n = batch.num_rows
            lo = max(row_start, rc)
            hi = min(row_end, rc+n)
            rc += n
            if lo >= hi: continue
            olo, ohi = lo-(rc-n), hi-(rc-n)
            X = np.column_stack([batch.column(c).to_numpy(
                zero_copy_only=False)[olo:ohi] for c in cols]).astype(np.float32)
            y = batch.column('is_fraud').to_numpy(
                zero_copy_only=False)[olo:ohi].astype(np.int8)
            yield X, y
    return gen

sig = (tf.TensorSpec(shape=(None, n_cols), dtype=tf.float32),
       tf.TensorSpec(shape=(None,),        dtype=tf.int8))

train_ds = tf.data.Dataset.from_generator(
    make_generator(train_path, 0, n_train, BATCH_SIZE, hoba_cols),
    output_signature=sig).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_generator(
    make_generator(train_path, n_train, n_rows_train, BATCH_SIZE, hoba_cols),
    output_signature=sig).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_generator(
    make_generator(test_path, 0, n_rows_test, BATCH_SIZE, hoba_cols),
    output_signature=sig).prefetch(tf.data.AUTOTUNE)

# ── BPNN on HOBA ─────────────────────────────────────────────
print("\nTraining BPNN on HOBA features (streaming)...")
bpnn_hoba = build_bpnn(input_dim=n_cols)
bpnn_hoba.fit(train_ds, validation_data=val_ds, epochs=20,
              class_weight=cw_hoba, callbacks=[early_stop], verbose=1)
print(f"RAM after fit: {psutil.virtual_memory().percent}%")

print("\nPredicting on test set...")
y_prob_list, y_test_list = [], []
for X_batch, y_batch in test_ds:
    y_prob_list.append(bpnn_hoba.predict(X_batch, verbose=0).flatten())
    y_test_list.append(y_batch.numpy())
y_prob = np.concatenate(y_prob_list)
y_test = np.concatenate(y_test_list)
y_pred = (y_prob >= 0.5).astype(int)
evaluate_model("BPNN — HOBA", y_test, y_pred, y_prob)

print("\n")
print_results_table()

Loading RFM...
RFM ready: (1035957, 65) | RAM: 19.9%

Training BPNN on RFM features...
Epoch 1/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - AUC: 0.9186 - loss: 0.3677 - val_AUC: 0.9613 - val_loss: 0.2453
Epoch 2/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - AUC: 0.9506 - loss: 0.2852 - val_AUC: 0.9688 - val_loss: 0.1981
Epoch 3/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - AUC: 0.9572 - loss: 0.2629 - val_AUC: 0.9714 - val_loss: 0.2039
Epoch 4/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - AUC: 0.9608 - loss: 0.2508 - val_AUC: 0.9712 - val_loss: 0.1926
Epoch 5/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - AUC: 0.9626 - loss: 0.2445 - val_AUC: 0.9727 - val_loss: 0.1849
Epoch 6/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - AUC: 0.9644 - loss: 0.2386 - val_AUC: 0.9743 - val_loss: 0.1710
Epoch 7/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - AUC: 0.9660 - loss: 0.2328 - val_AUC: 0.9731 - val_loss: 0.1787
Epoch 8/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - AUC: 0.9664 - lo